# sms_v2 — two-stage SMS benchmark (Stage 1 classifiers → Stage 2 extraction)

Runs the whole benchmark on a Colab **T4**:

| stage | what | models |
|---|---|---|
| Stage 1 | is this SMS a successful transaction? Experiment **A** binary, **B** 6-class | TF-IDF + LR / Linear SVM / Multinomial NB, MARBERT, AraBERT, XLM-RoBERTa, v1's own gate (reference) |
| Stage 2 | extract amount, currency, counterparty, date, direction, account, … | the existing v1 parser, unchanged |
| End to end | Stage 1 errors counted as missed / false records | every finished classifier |

**Runtime → Change runtime type → T4 GPU** before you start.

**Reliability.** Everything the benchmark produces goes to Google Drive. Every run writes a `DONE.json`
marker last; a finished run is **skipped** on re-run, never overwritten unless `OVERWRITE = True`. Transformers
checkpoint **after every epoch** to the VM's local disk and copy that state to Drive (`DRIVE_ROOT/resume/`).
**After a disconnect (or a restarted runtime): run sections 0 and 1, then any section you want** — or simply
Run all. Section 1 restores everything the later cells need. A transformer run continues from the newest copy that verifies —
local if the VM survived, otherwise Drive, falling back to the previous epoch if the last Drive copy never
finished uploading. Each transformer trains in its **own process**, so its GPU memory is fully released before
the next.

Splits, labels and results are documented in `sms_v2/BENCHMARK.md`.

## 0. Configuration — the only cell you should need to edit

In [ ]:
DRIVE_ROOT = '/content/drive/MyDrive/masroofy_sms_v2'   # results + checkpoints live here
CODE_SOURCE = 'drive'        # 'drive'  : DRIVE_ROOT/code/sms_colab.tgz  (made by `python pack_for_colab.py`)
                             # 'upload' : pick sms_colab.tgz in the browser
                             # 'git'    : clone BRANCH from REPO_URL (committed + pushed code only)
REPO_URL = 'https://github.com/<you>/Masroofy.git'      # used when CODE_SOURCE == 'git'
BRANCH = 'sms'

SPLITS = ['template']        # primary. Add 'random' (optimistic) / 'sender' (unseen bank) if time allows
EXPERIMENTS = ['binary', 'multiclass']
SKLEARN_MODELS = ['tfidf_lr', 'tfidf_svm', 'tfidf_nb', 'v1_gate']
TRANSFORMER_MODELS = ['marbert', 'arabert', 'xlm_roberta']   # trained one at a time, in this order

OVERWRITE = False            # True = redo runs that already finished (their old results are deleted)
TRANSFORMER_OVERRIDE = None  # e.g. {'epochs': 1, 'batch_size': 16} for a quick look; None = config/settings.py

# Disconnect protection for transformer training (see section 5a):
BACKUP_TO_DRIVE = True       # copy the resumable state to DRIVE_ROOT/resume after each epoch
BACKUP_SLOTS = 2             # 2: a disconnect DURING a backup falls back one epoch (~9 GB on Drive for XLM-R)
                             # 1: half the space; a disconnect during a backup restarts that run
BACKUP_EVERY = 1             # epochs between backups (the last epoch is always backed up)

## 1. Setup — run this after EVERY connect, reconnect or runtime restart

Nothing in Python memory survives a disconnect, so this one cell rebuilds all of it: mounts Drive, points the
benchmark at Drive (results, Drive backup) and the local disk (checkpoints), fetches the code, checks the GPU,
and defines the helpers and imports every later cell uses. It is safe to run any number of times. After it,
any section can be run on its own and in any order: everything a section needs from earlier work is read
from files on Drive, not from variables.

**Getting the code** (fetched **fresh every time this cell runs**, so after editing on your PC you just re-pack
and re-run this cell; results on Drive and checkpoints in `/content/checkpoints` are outside the code folder):

* `drive` (no commit needed): on your PC, in `sms_v2/`, run `python pack_for_colab.py` and copy the
  `sms_colab.tgz` it writes to `DRIVE_ROOT/code/` (with Google Drive for desktop:
  `python pack_for_colab.py --to "G:/My Drive/masroofy_sms_v2/code"`). Give Drive a minute to sync.
* `upload` (no commit needed): same file, picked in the browser when the cell asks.
* `git`: the code must be committed and pushed on `BRANCH`.

The printed `CODE_VERSION` (pack time, content hash, uncommitted files) is saved with every run's metrics.

Changing a hyperparameter in `config/settings.py` while a transformer run is half done makes its resume
refuse (the checkpoint belongs to the old configuration): finish it first, or re-run it with `OVERWRITE`.

In [ ]:
import os, sys, json, time, glob, shutil, subprocess, importlib.util
import pandas as pd
from IPython.display import Markdown, Image, display

assert 'DRIVE_ROOT' in globals(), 'Run section 0 (configuration) first.'

# ---- Drive + where everything is written (read by config/settings.py in every bench.py process)
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
os.makedirs(DRIVE_ROOT, exist_ok=True)
os.environ['SMS_V2_OUTPUT'] = os.path.join(DRIVE_ROOT, 'outputs')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
# per-epoch checkpoints on the VM's fast local disk (rewriting them on Drive every epoch would fill its Trash)
os.environ['SMS_V2_CHECKPOINTS'] = '/content/checkpoints'
if BACKUP_TO_DRIVE:
    os.environ['SMS_V2_CHECKPOINT_BACKUP'] = os.path.join(DRIVE_ROOT, 'resume')
    os.environ['SMS_V2_BACKUP_SLOTS'] = str(BACKUP_SLOTS)
    os.environ['SMS_V2_BACKUP_EVERY'] = str(BACKUP_EVERY)
else:
    os.environ.pop('SMS_V2_CHECKPOINT_BACKUP', None)

# ---- code, fresh every time
WORK = '/content/work'
os.chdir('/content')
shutil.rmtree(WORK, ignore_errors=True)
os.makedirs(WORK)
if CODE_SOURCE == 'drive':
    tgz = f'{DRIVE_ROOT}/code/sms_colab.tgz'
    if not os.path.exists(tgz):
        raise FileNotFoundError(f'{tgz} not found: run `python pack_for_colab.py` on your PC and copy '
                                'sms_colab.tgz there (or use CODE_SOURCE = "upload")')
    subprocess.run(['tar', '-xzf', tgz, '-C', WORK], check=True)
elif CODE_SOURCE == 'upload':
    from google.colab import files
    up = files.upload()
    subprocess.run(['tar', '-xzf', next(iter(up)), '-C', WORK], check=True)
elif CODE_SOURCE == 'git':
    subprocess.run(['git', 'clone', '--depth', '1', '-b', BRANCH, REPO_URL, WORK + '/repo'], check=True)
    for d in ('sms_v2', 'sms_eval_final'):
        shutil.move(f'{WORK}/repo/{d}', f'{WORK}/{d}')
else:
    raise ValueError(CODE_SOURCE)
os.chdir(f'{WORK}/sms_v2')
if f'{WORK}/sms_v2' not in sys.path:
    sys.path.insert(0, f'{WORK}/sms_v2')
for m in [m for m in sys.modules if m.split('.')[0] in ('benchmark', 'classification', 'config',
                                                         'evaluation', 'extraction', 'sms_pipeline')]:
    del sys.modules[m]                      # a re-run must import the code just fetched, not the old one

# ---- dependencies + GPU
if importlib.util.find_spec('transformers') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers'], check=True)
import torch, transformers, sklearn
GPU = torch.cuda.is_available()

# ---- helpers used by the later sections
def bench(*args):
    """Run bench.py in its own process, streaming its output. -> return code."""
    p = subprocess.Popen([sys.executable, '-u', 'bench.py', *args], stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if 'Warning' not in line and 'warnings.warn' not in line:
            print(line, end='')
    return p.wait()

ow = ['--overwrite'] if OVERWRITE else []

# ---- report
ver = 'CODE_VERSION.json'
if os.path.exists(ver):
    v = json.load(open(ver))
    print(f"code        -> {v['content_hash']}  packed {v['packed']}  from {v['git_branch']} @ {v['git_commit']}"
          f"  ({len(v['uncommitted'])} uncommitted files)")
else:
    print('code        ->', subprocess.run(['git', '-C', WORK + '/repo', 'rev-parse', '--short', 'HEAD'],
                                           capture_output=True, text=True).stdout.strip() or BRANCH, '(git)')
print('results     ->', os.environ['SMS_V2_OUTPUT'])
print('checkpoints ->', os.environ['SMS_V2_CHECKPOINTS'], '(local disk)')
print('backup      ->', os.environ.get('SMS_V2_CHECKPOINT_BACKUP', 'OFF'))
print('torch', torch.__version__, '| transformers', transformers.__version__, '| sklearn', sklearn.__version__)
print('GPU         ->', torch.cuda.get_device_name(0) if GPU else
      'NONE (TF-IDF still runs; transformers would be very slow: Runtime -> Change runtime type -> T4 GPU)')
if GPU:
    free, total = torch.cuda.mem_get_info()
    print(f'VRAM {total / 2**30:.1f} GB, free {free / 2**30:.1f} GB')
done = len(glob.glob(os.path.join(os.environ['SMS_V2_OUTPUT'], 'results', '*', '*', '*', 'DONE.json')))
print(f'\nalready finished on Drive: {done} classifier runs (they will be skipped). Ready.')

## 2. Labels, class distributions and splits

`prepare` validates the labels against the gold data, writes `data/processed/label_audit.csv` (every derived
label with the rule that produced it) and **loads** the committed split files in `data/splits/` — it does not
re-draw them, so Colab and the local machine evaluate on identical rows. A changed corpus is detected by hash.

In [ ]:
assert 'bench' in globals(), 'Run sections 0 and 1 first (after every connect / disconnect).'
splits_arg = ','.join(SPLITS)
!python bench.py prepare --splits $splits_arg

In [ ]:
assert 'bench' in globals(), 'Run sections 0 and 1 first (after every connect / disconnect).'
import json, pandas as pd
rep = json.load(open('data/processed/dataset_report.json', encoding='utf-8'))
print('records:', rep['n_records'], '| data hash:', rep['data_hash'])
display(pd.Series(rep['multiclass'], name='multi-class').to_frame().join(
        pd.Series(rep['binary'], name='binary'), how='outer'))
for s, d in rep['splits'].items():
    print(f'\nsplit {s}  grouping: {d["grouping"]}')
    display(pd.DataFrame({k: v['multiclass'] for k, v in d['summary'].items()}).fillna(0).astype(int))
audit = pd.read_csv('data/processed/label_audit.csv')
print('\nhow each multi-class label was derived:')
display(audit.groupby(['label_multi', 'label_reason']).size().rename('n').reset_index()
        .sort_values(['label_multi', 'n'], ascending=[True, False]))

In [ ]:
assert 'bench' in globals(), 'Run sections 0 and 1 first (after every connect / disconnect).'
# keep a copy of the exact split files next to the results, for the record
import shutil, filecmp
dst = os.path.join(os.environ['SMS_V2_OUTPUT'], 'splits')
os.makedirs(dst, exist_ok=True)
for s in SPLITS:
    src, out = f'data/splits/{s}.json', os.path.join(dst, f'{s}.json')
    if os.path.exists(out) and not filecmp.cmp(src, out, shallow=False):
        raise RuntimeError(f'{out} differs from the repo split: results on Drive were made on another split. '
                           'Use a new DRIVE_ROOT or delete the old results deliberately.')
    shutil.copy(src, out)
print('splits recorded in', dst)

## 3. Quick tests (CPU, ~2 min)

Labels, leakage, every classifier interface, routing rules, end-to-end accounting, and the transformer train → interrupt → resume → save → load cycle on a tiny model.

In [ ]:
assert 'bench' in globals(), 'Run sections 0 and 1 first (after every connect / disconnect).'
!python -m pytest tests -q -p no:cacheprovider 2>&1 | tail -5

## 4. Traditional baselines (CPU, ~1 min per split)

In [ ]:
assert 'bench' in globals(), 'Run sections 0 and 1 first (after every connect / disconnect).'
rc = bench('classify', '--models', ','.join(SKLEARN_MODELS), '--experiments', ','.join(EXPERIMENTS),
           '--splits', ','.join(SPLITS), *ow)
print('\nexit code', rc, '(non-zero = a run failed; it is in experiment_log.jsonl and the status table)')

## 5. Transformers, one at a time

Each (split, experiment, model) is a **separate process**: when it ends, all of its GPU memory is returned, so
two models are never in VRAM together. Interrupted? Re-run this cell: finished runs are skipped and an
interrupted run resumes from its newest saved epoch (local disk, or the Drive backup if the VM was lost). A failed run is reported and the loop moves on.

Roughly 1–3 min per epoch per model on a T4 with ~1.5k training messages; XLM-R is the slowest.

### 5a. Backup slots and Drive space

What the Drive backup holds right now. After a disconnect, a slot in state `ok` is what the interrupted run
continues from; `free` slots belong to finished runs and are overwritten by the next one (never deleted, so
nothing lands in Drive's Trash). If the free-space line drops below ~10 GB, set `BACKUP_SLOTS = 1`.

In [ ]:
assert 'bench' in globals(), 'Run sections 0 and 1 first (after every connect / disconnect).'
import shutil, pandas as pd
from classification.checkpoint_backup import describe
u = shutil.disk_usage('/content/drive/MyDrive')
print(f'Drive: {u.free / 2**30:.1f} GB free of {u.total / 2**30:.1f} GB (as reported by the Drive mount)')
u = shutil.disk_usage('/content')
print(f'VM disk: {u.free / 2**30:.1f} GB free')
if os.environ.get('SMS_V2_CHECKPOINT_BACKUP'):
    display(pd.DataFrame(describe(os.environ['SMS_V2_CHECKPOINT_BACKUP'], BACKUP_SLOTS)))
else:
    print('Drive backup is OFF: a lost VM restarts an unfinished transformer run from epoch 0.')

### 5b. Train

In [ ]:
assert 'bench' in globals(), 'Run sections 0 and 1 first (after every connect / disconnect).'
import json, time
failed = []
for split in SPLITS:
    for exp in EXPERIMENTS:
        for model in TRANSFORMER_MODELS:
            t0 = time.time()
            args = ['classify', '--models', model, '--experiments', exp, '--splits', split, *ow]
            if TRANSFORMER_OVERRIDE:
                args += ['--override', json.dumps(TRANSFORMER_OVERRIDE)]
            rc = bench(*args)
            print(f'--- {split}/{exp}/{model}: exit {rc} after {(time.time() - t0) / 60:.1f} min\n')
            if rc != 0:
                failed.append((split, exp, model))
            !nvidia-smi --query-gpu=memory.used,memory.total --format=csv,noheader
print('FAILED:', failed or 'none')
# Drive uploads in the background: give it a moment to finish before you close the tab
if hasattr(os, 'sync'):
    os.sync()

## 6. Standalone extraction, then the end-to-end pipeline

Extraction is fitted on the training split and scored on gold transactions only (a perfect Stage 1). End to end then counts every Stage-1 miss and false accept.

In [ ]:
assert 'bench' in globals(), 'Run sections 0 and 1 first (after every connect / disconnect).'
bench('extract', '--splits', ','.join(SPLITS), *ow)
bench('e2e', '--splits', ','.join(SPLITS), '--experiments', ','.join(EXPERIMENTS))

## 7. Comparison tables and charts

Written to `outputs/results/<split>/summary/` on Drive: CSV + JSON tables, `REPORT.md`, `charts/`, `confusion_matrices/`.

In [ ]:
assert 'bench' in globals(), 'Run sections 0 and 1 first (after every connect / disconnect).'
bench('compare', '--splits', ','.join(SPLITS))
from IPython.display import Markdown, Image, display
import glob
for split in SPLITS:
    base = os.path.join(os.environ['SMS_V2_OUTPUT'], 'results', split, 'summary')
    display(Markdown(open(os.path.join(base, 'REPORT.md'), encoding='utf-8').read()))
    for png in sorted(glob.glob(os.path.join(base, 'charts', '*.png'))):
        display(Image(png, width=760))

In [ ]:
assert 'bench' in globals(), 'Run sections 0 and 1 first (after every connect / disconnect).'
# confusion matrices
for split in SPLITS:
    for png in sorted(glob.glob(os.path.join(os.environ['SMS_V2_OUTPUT'], 'results', split,
                                             'summary', 'confusion_matrices', '*.png'))):
        display(Image(png, width=380))

## 8. Run status and experiment log

Nothing incomplete hides: failed and interrupted runs are listed here.

In [ ]:
assert 'bench' in globals(), 'Run sections 0 and 1 first (after every connect / disconnect).'
bench('status', '--splits', ','.join(SPLITS))
log = [json.loads(l) for l in open(os.path.join(os.environ['SMS_V2_OUTPUT'], 'experiment_log.jsonl'), encoding='utf-8')]
bad = [e for e in log if e['event'] in ('failed', 'interrupted')]
print(f'{len(log)} log events, {len(bad)} failed/interrupted')
for e in bad[-10:]:
    print(e['time'], e['event'], e.get('split'), e.get('experiment'), e.get('model'), e.get('error', ''))

## 9. Use the pipeline

Pick any finished classifier; the extractor stays the same.

In [ ]:
assert 'bench' in globals(), 'Run sections 0 and 1 first (after every connect / disconnect).'
from sms_pipeline import SmsPipeline
OUT = os.environ['SMS_V2_OUTPUT']
pipe = SmsPipeline.from_dirs(f'{OUT}/models/{SPLITS[0]}/multiclass/tfidf_svm/final',
                             f'{OUT}/models/{SPLITS[0]}/extractor/v1_sklearn')
for sms in ['تم خصم 250.00 جنيه من حسابك ****4321 لدى CARREFOUR MAADI يوم 10/08/2026',
            'Your CIB OTP is 847237. Do not share this code with anyone.',
            'تم رفض المعاملة على بطاقتك المنتهية بـ **4832 بقيمة 191.00 EGP لعدم كفاية الرصيد']:
    print(json.dumps(pipe.process(sms), ensure_ascii=False, indent=1))

---
# Experiment 1 — classifier × extractor combinations (AraBERT / MARBERT)

Four end-to-end configurations per classification experiment (A binary, B 6-class) = **8 configurations**:

| | AraBERT extractor | MARBERT extractor |
|---|---|---|
| **AraBERT classifier** | AraBERT → AraBERT | AraBERT → MARBERT |
| **MARBERT classifier** | MARBERT → AraBERT | MARBERT → MARBERT |

**Needed first**
* the classifier runs of section 5 (`arabert` / `marbert`, binary and/or 6-class) — reused, never retrained;
* the two **transformer extractors** (section E1.1): they did not exist before. Architecture = `marbert.py`'s
  extractor (encoder + BIO token head + type head + channel head), trained on the gold transactions of the same
  train split, early-stopped on validation. One training per encoder; GPU, ~same cost as one classifier.

Each combination cell is cheap (no training): it scores *routed by this classifier* × *extracted by this
extractor* on the test set, and runs the **real pipeline one SMS at a time** on 50 test messages for latency
(and checks it reproduces the cached result). A missing model is reported with the command that trains it.
Results: `outputs/results/<split>/extractors/`, `.../combos/<experiment>/<classifier>__<extractor>/`.

In [ ]:
assert 'bench' in globals(), 'Run sections 0 and 1 first (after every connect / disconnect).'
SPL = globals().get('SPLIT') or SPLITS[0]

# E1.0 What is done so far for the new experiments (reads Drive; trains nothing)
bench('compare-new', '--splits', SPL, '--no-charts')
display(pd.read_csv(os.path.join(os.environ['SMS_V2_OUTPUT'], 'results', SPL, 'summary_new', 'status.csv')))

### E1.1 Train the transformer extractors (GPU, one at a time)

In [ ]:
assert 'bench' in globals(), 'Run sections 0 and 1 first (after every connect / disconnect).'
SPL = globals().get('SPLIT') or SPLITS[0]
overwrite = False    # True = redo this finished run (its old result is deleted)

# E1.1a MARBERT extractor
override = None      # e.g. {'epochs': 1} for a quick test; None = config/settings.py
bench('extract-tf', '--models', 'marbert', '--splits', SPL, *(['--overwrite'] if overwrite else []),
      *(['--override', json.dumps(override)] if override else []))

In [ ]:
assert 'bench' in globals(), 'Run sections 0 and 1 first (after every connect / disconnect).'
SPL = globals().get('SPLIT') or SPLITS[0]
overwrite = False    # True = redo this finished run (its old result is deleted)

# E1.1b AraBERT extractor
override = None      # e.g. {'epochs': 1} for a quick test; None = config/settings.py
bench('extract-tf', '--models', 'arabert', '--splits', SPL, *(['--overwrite'] if overwrite else []),
      *(['--override', json.dumps(override)] if override else []))

### E1.2 Combinations — Experiment A (binary classifier)

In [ ]:
assert 'bench' in globals(), 'Run sections 0 and 1 first (after every connect / disconnect).'
SPL = globals().get('SPLIT') or SPLITS[0]
overwrite = False    # True = redo this finished run (its old result is deleted)

# A: arabert classifier -> arabert extractor
bench('combos', '--experiments', 'binary', '--classifiers', 'arabert', '--extractors', 'arabert', '--splits', SPL,
      *(['--overwrite'] if overwrite else []))

In [ ]:
assert 'bench' in globals(), 'Run sections 0 and 1 first (after every connect / disconnect).'
SPL = globals().get('SPLIT') or SPLITS[0]
overwrite = False    # True = redo this finished run (its old result is deleted)

# A: arabert classifier -> marbert extractor
bench('combos', '--experiments', 'binary', '--classifiers', 'arabert', '--extractors', 'marbert', '--splits', SPL,
      *(['--overwrite'] if overwrite else []))

In [ ]:
assert 'bench' in globals(), 'Run sections 0 and 1 first (after every connect / disconnect).'
SPL = globals().get('SPLIT') or SPLITS[0]
overwrite = False    # True = redo this finished run (its old result is deleted)

# A: marbert classifier -> arabert extractor
bench('combos', '--experiments', 'binary', '--classifiers', 'marbert', '--extractors', 'arabert', '--splits', SPL,
      *(['--overwrite'] if overwrite else []))

In [ ]:
assert 'bench' in globals(), 'Run sections 0 and 1 first (after every connect / disconnect).'
SPL = globals().get('SPLIT') or SPLITS[0]
overwrite = False    # True = redo this finished run (its old result is deleted)

# A: marbert classifier -> marbert extractor
bench('combos', '--experiments', 'binary', '--classifiers', 'marbert', '--extractors', 'marbert', '--splits', SPL,
      *(['--overwrite'] if overwrite else []))

### E1.3 Combinations — Experiment B (6-class classifier)

In [ ]:
assert 'bench' in globals(), 'Run sections 0 and 1 first (after every connect / disconnect).'
SPL = globals().get('SPLIT') or SPLITS[0]
overwrite = False    # True = redo this finished run (its old result is deleted)

# B: arabert classifier -> arabert extractor
bench('combos', '--experiments', 'multiclass', '--classifiers', 'arabert', '--extractors', 'arabert', '--splits', SPL,
      *(['--overwrite'] if overwrite else []))

In [ ]:
assert 'bench' in globals(), 'Run sections 0 and 1 first (after every connect / disconnect).'
SPL = globals().get('SPLIT') or SPLITS[0]
overwrite = False    # True = redo this finished run (its old result is deleted)

# B: arabert classifier -> marbert extractor
bench('combos', '--experiments', 'multiclass', '--classifiers', 'arabert', '--extractors', 'marbert', '--splits', SPL,
      *(['--overwrite'] if overwrite else []))

In [ ]:
assert 'bench' in globals(), 'Run sections 0 and 1 first (after every connect / disconnect).'
SPL = globals().get('SPLIT') or SPLITS[0]
overwrite = False    # True = redo this finished run (its old result is deleted)

# B: marbert classifier -> arabert extractor
bench('combos', '--experiments', 'multiclass', '--classifiers', 'marbert', '--extractors', 'arabert', '--splits', SPL,
      *(['--overwrite'] if overwrite else []))

In [ ]:
assert 'bench' in globals(), 'Run sections 0 and 1 first (after every connect / disconnect).'
SPL = globals().get('SPLIT') or SPLITS[0]
overwrite = False    # True = redo this finished run (its old result is deleted)

# B: marbert classifier -> marbert extractor
bench('combos', '--experiments', 'multiclass', '--classifiers', 'marbert', '--extractors', 'marbert', '--splits', SPL,
      *(['--overwrite'] if overwrite else []))

### E1.4 One SMS in, final JSON out (any finished combination)

In [ ]:
assert 'bench' in globals(), 'Run sections 0 and 1 first (after every connect / disconnect).'
SPL = globals().get('SPLIT') or SPLITS[0]

from sms_pipeline import SmsPipeline
pipe = SmsPipeline.from_combo(SPL, 'binary', 'arabert', 'marbert')   # experiment, classifier, extractor
for sms in ['تم خصم 250.00 جنيه من حسابك ****4321 لدى CARREFOUR MAADI يوم 10/08/2026',
            'Your CIB OTP is 847237. Do not share this code with anyone.']:
    print(json.dumps(pipe.process(sms), ensure_ascii=False, indent=1))

---
# Experiment 2 — one joint model (classification + extraction in one workflow)

The runs above use **two models** (classifier, then extractor). Here **one** AraBERT or MARBERT has a classification
head (binary or 6-class: the same gold labels as section 5, no new classes) **and** the extractor's heads on one
shared encoder, trained together. One forward pass returns the class and the record. Loss = class-weighted CE +
the extraction losses (gold transactions only). Early stopping on mean(val macro-F1, val full-record exact).
New training per cell (GPU). Results: `outputs/results/<split>/joint/<experiment>/<model>/`.

In [ ]:
assert 'bench' in globals(), 'Run sections 0 and 1 first (after every connect / disconnect).'
SPL = globals().get('SPLIT') or SPLITS[0]
overwrite = False    # True = redo this finished run (its old result is deleted)

# E2 joint arabert — A: binary class head
override = None      # e.g. {'epochs': 1} for a quick test; None = config/settings.py
bench('joint', '--models', 'arabert', '--experiments', 'binary', '--splits', SPL,
      *(['--overwrite'] if overwrite else []), *(['--override', json.dumps(override)] if override else []))

In [ ]:
assert 'bench' in globals(), 'Run sections 0 and 1 first (after every connect / disconnect).'
SPL = globals().get('SPLIT') or SPLITS[0]
overwrite = False    # True = redo this finished run (its old result is deleted)

# E2 joint arabert — B: multiclass class head
override = None      # e.g. {'epochs': 1} for a quick test; None = config/settings.py
bench('joint', '--models', 'arabert', '--experiments', 'multiclass', '--splits', SPL,
      *(['--overwrite'] if overwrite else []), *(['--override', json.dumps(override)] if override else []))

In [ ]:
assert 'bench' in globals(), 'Run sections 0 and 1 first (after every connect / disconnect).'
SPL = globals().get('SPLIT') or SPLITS[0]
overwrite = False    # True = redo this finished run (its old result is deleted)

# E2 joint marbert — A: binary class head
override = None      # e.g. {'epochs': 1} for a quick test; None = config/settings.py
bench('joint', '--models', 'marbert', '--experiments', 'binary', '--splits', SPL,
      *(['--overwrite'] if overwrite else []), *(['--override', json.dumps(override)] if override else []))

In [ ]:
assert 'bench' in globals(), 'Run sections 0 and 1 first (after every connect / disconnect).'
SPL = globals().get('SPLIT') or SPLITS[0]
overwrite = False    # True = redo this finished run (its old result is deleted)

# E2 joint marbert — B: multiclass class head
override = None      # e.g. {'epochs': 1} for a quick test; None = config/settings.py
bench('joint', '--models', 'marbert', '--experiments', 'multiclass', '--splits', SPL,
      *(['--overwrite'] if overwrite else []), *(['--override', json.dumps(override)] if override else []))

In [ ]:
assert 'bench' in globals(), 'Run sections 0 and 1 first (after every connect / disconnect).'
SPL = globals().get('SPLIT') or SPLITS[0]

# E2 one SMS in, final JSON out, ONE model
from sms_pipeline import JointPipeline
pipe = JointPipeline.from_run(SPL, 'multiclass', 'arabert')         # experiment, model
for sms in ['تم خصم 250.00 جنيه من حسابك ****4321 لدى CARREFOUR MAADI يوم 10/08/2026',
            'Your CIB OTP is 847237. Do not share this code with anyone.']:
    print(json.dumps(pipe.process(sms), ensure_ascii=False, indent=1))

### E3. Comparison: Experiment 1 vs Experiment 2 vs the existing two-stage runs

Writes `outputs/results/<split>/summary_new/` (tables, `REPORT_NEW.md`, charts, confusion matrices). The existing
`summary/` is not touched. Re-run any time; it only reads finished runs.

In [ ]:
assert 'bench' in globals(), 'Run sections 0 and 1 first (after every connect / disconnect).'
SPL = globals().get('SPLIT') or SPLITS[0]

bench('compare-new', '--splits', SPL)
base = os.path.join(os.environ['SMS_V2_OUTPUT'], 'results', SPL, 'summary_new')
display(Markdown(open(os.path.join(base, 'REPORT_NEW.md'), encoding='utf-8').read()))
for png in sorted(glob.glob(os.path.join(base, 'charts', '*.png'))):
    display(Image(png, width=760))
for png in sorted(glob.glob(os.path.join(base, 'confusion_matrices', '*.png'))):
    display(Image(png, width=380))

---
# Appendix — the original v1 vs v2 (MARBERT extractor) comparison

The cells below are the notebook as it was before the benchmark was added, unchanged. They train the v2
MARBERT **gate + extractor** with `train.py` on `dataset.py`'s four splits and compare against v1 with
`compare.py`. They run in `sms_v2/` and write `runs/` on the Colab disk (not Drive). Their splits are
`dataset.py`'s, **not** the benchmark's, so their numbers are not comparable with sections 4–7.

## 2. Install and check the GPU

Colab ships torch already; only `transformers` is missing.

In [ ]:
!pip install -q transformers
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
assert torch.cuda.is_available(), 'Runtime -> Change runtime type -> T4 GPU, then re-run'

## 3. Sanity checks that need no model

`test_spans.py` asserts the REAL_FORMATS.md failure list; `test_labels.py` prints
v2's extraction ceiling. If either of these is unhappy, training is a waste of time.

In [ ]:
!python dataset.py splits
!python test_spans.py | tail -3
!python test_labels.py | head -14

## 4. Smoke test

64 records, 1 epoch. Downloads MARBERT's ~650 MB of weights on the way, so the
first run is slower than the training itself. Fail fast before the real run.

In [ ]:
!python train.py --split random --stage both --limit 64 --epochs 1 --out runs/_smoke
!python predict.py --run runs/_smoke "تم خصم مبلغ 250.00 جنيه من حسابك المنتهي بـ 4321 لدى CARREFOUR" --confidence

## 5. Train for real

Four splits × (gate + extractor). On a T4 expect roughly 4–8 min per split at 3
epochs. Drop splits from the list to go faster — `random` and `real_formats` are
the two that matter most (most optimistic and most production-like).

In [ ]:
SPLITS = ['random', 'unseen_wording', 'unseen_sender', 'real_formats']

import subprocess, time
for s in SPLITS:
    t0 = time.time()
    print(f'\n{"=" * 60}\n=== {s}\n{"=" * 60}', flush=True)
    subprocess.run(['python', 'train.py', '--split', s, '--stage', 'both',
                    '--epochs', '3', '--batch-size', '16'], check=True)
    print(f'--- {s} done in {(time.time() - t0) / 60:.1f} min')

## 6. Compare

Trains v1 on the same rows, scores both with v1's own `score()`, writes the report.

In [ ]:
# SPLITS comes from the training cell; this default lets you run compare on its own.
try:
    SPLITS
except NameError:
    SPLITS = ['random', 'unseen_wording', 'unseen_sender', 'real_formats']
splits_arg = ','.join(SPLITS)

!python compare.py --splits $splits_arg

In [ ]:
# A compact headline table out of the json
import json
r = json.load(open('compare_results.json', encoding='utf-8'))
keys = ['tx_f1', 'non_tx_rejected', 'full_record_exact', 'merchant', 'datetime',
        'amount', 'account_last4', 'type']
for split, d in r.items():
    print(f'\n{split}  (n={d["n_test"]})')
    print(f'  {"metric":20s} {"v1":>8s} {"v2":>8s} {"delta":>8s}')
    for k in keys:
        a, b = d['v1'].get(k), (d['v2'] or {}).get(k)
        if a is None or b is None:
            continue
        print(f'  {k:20s} {a:8.3f} {b:8.3f} {b - a:+8.3f}')

## 7. Bring the results home

The report is small. The checkpoints are ~650 MB per stage per split, so only take
them if you intend to serve the model — otherwise keep the report and retrain when
you need to.

In [ ]:
from google.colab import files
files.download('compare_report.txt')
files.download('compare_results.json')

In [ ]:
# Optional: the checkpoints, one split at a time (big)
# !tar -czf runs_random.tgz runs/random && ls -lh runs_random.tgz
# files.download('runs_random.tgz')

## Knobs worth turning

| flag | why |
|---|---|
| `--model-name UBC-NLP/MARBERTv2` | newer MARBERT, usually a little better |
| `--model-name xlm-roberta-base` | ~40% of this corpus is English or mixed, and MARBERT's vocabulary is Arabic-first — this is the obvious thing to test against it |
| `--model-name aubmindlab/bert-base-arabertv02` | AraBERT, trained on MSA rather than dialect |
| `--epochs 5` | 3 is a guess; watch the loss trace |
| `--batch-size 32` | fits a T4 at `--max-len 160` and is faster |
| `--backend linear` | linear gate + MARBERT extractor, to see which stage the win comes from |

Each `--model-name` run writes to its own `--out` directory, so you can compare
encoders the same way `compare.py` compares v1 against v2.